# 02 — Exploratory Data Analysis (Cleaned Data)

This notebook explores the **cleaned** heart failure readmission dataset to understand it before
statistical testing and modeling. It runs **after `01_cleaning.ipynb`**, which writes
`data/processed/cleaned.csv`.

What this notebook covers (the project's required EDA):
- **Dataset shape** — how many patients and columns we are working with.
- **Target balance** — how many patients were readmitted, and the majority-class baseline to beat.
- **Missingness** — which values are missing and will need imputation during modeling.
- **Feature relationships** — how the key features (`adherence_score`, `bnp`, `age`,
  `ace_inhibitor`, `beta_blocker`) relate to readmission.
- A correlation check and binned risk gradients to size up how much signal the data holds.

Everything here is **descriptive**: it shows the direction and size of associations, not causation.
Formal hypothesis tests live in `03_statistical_testing.ipynb`.

## Setup and load the cleaned data

We add the project root to the path so `from src... import ...` works, then load the cleaned file
produced by notebook 01. If it is missing, run `01_cleaning.ipynb` first.

In [ ]:
import sys
from pathlib import Path

# Make the project's `src` package importable from inside the notebooks/ folder.
sys.path.insert(0, str(Path.cwd().parent))
%matplotlib inline

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Shared constants: target column name and output folders (defined once in src/config.py).
from src.config import TARGET, PROCESSED_DIR, FIGURES_DIR, TABLES_DIR

sns.set_theme(style="whitegrid")  # consistent look for every plot below

In [ ]:
# Load the cleaned dataset. Stop early with a clear message if notebook 01 has not been run.
cleaned_path = PROCESSED_DIR / "cleaned.csv"
if not cleaned_path.exists():
    raise FileNotFoundError(
        f"{cleaned_path} was not found. Run 01_cleaning.ipynb first to create cleaned.csv."
    )

df = pd.read_csv(cleaned_path)

# Dataset shape: rows = patients, columns = features + target.
print("Rows, columns:", df.shape)
df.head()

## Column overview

We sort the columns into three groups so each gets the right kind of summary and plot later:
- **numeric** — continuous measurements (age, labs, scores, ...),
- **binary** — 0/1 treatment flags (kept separate from real measurements),
- **categorical** — text labels like `gender` and `income_level`.

In [ ]:
df.info()  # dtypes and non-null counts at a glance

In [ ]:
# Start from all numeric columns, drop the target, then peel off the 0/1 flags.
numeric = df.select_dtypes("number").drop(columns=[TARGET]).columns.tolist()
binary = [c for c in numeric if df[c].dropna().isin([0, 1]).all()]  # flags only hold 0 or 1
numeric = [c for c in numeric if c not in binary]                   # keep the truly continuous ones
categorical = df.select_dtypes(exclude="number").columns.tolist() + binary

print("numeric    :", numeric)
print("binary     :", binary)
print("categorical:", categorical)

## Target balance and the baseline to beat

`readmitted_30d` is the outcome: 1 = readmitted within 30 days, 0 = not. The **majority-class
baseline** is the accuracy a model gets by always predicting the most common class. Every real
model has to beat this number, so we compute it up front.

In [ ]:
# Counts and proportions for each class.
counts = df[TARGET].value_counts().sort_index()
proportions = df[TARGET].value_counts(normalize=True).sort_index()
balance = pd.DataFrame({"count": counts, "proportion": proportions.round(3)})

# Majority-class baseline = proportion of the most common class (here, "not readmitted").
baseline = proportions.max()
print(balance)
print(f"\nMajority-class baseline accuracy: {baseline:.1%} (always predict class {proportions.idxmax()})")

In [ ]:
# Required plot: target balance bar chart.
fig, ax = plt.subplots(figsize=(5, 4))
counts.plot.bar(ax=ax, color=["steelblue", "indianred"])
ax.set_xlabel("readmitted_30d")
ax.set_ylabel("patients")
ax.set_title("Target balance: readmitted vs not")
ax.set_xticklabels(["not readmitted (0)", "readmitted (1)"], rotation=0)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_target_balance.png", dpi=120)

## Missingness (what the model will have to impute)

This is the missingness **after cleaning**. It is slightly larger than in the raw file because
`01_cleaning.ipynb` converted clinically impossible readings (e.g. negative creatinine) into `NaN`.
We do **not** fill the gaps here — imputation happens inside the modeling pipeline, after the
train/test split, to avoid data leakage. See notebook 01 for the before/after cleaning story.

In [ ]:
# Count and percent missing, keeping only columns that actually have gaps.
missing = pd.DataFrame({
    "missing": df.isna().sum(),
    "pct": (df.isna().mean() * 100).round(2),
})
missing = missing[missing["missing"] > 0].sort_values("missing", ascending=False)
missing

In [ ]:
# Required plot: missingness chart.
fig, ax = plt.subplots(figsize=(6, 4))
missing["pct"].sort_values().plot.barh(ax=ax, color="darkorange")
ax.set_xlabel("% missing")
ax.set_title("Missing values by column (after cleaning)")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_missingness.png", dpi=120)

## Numeric feature distributions

A quick look at the shape of each continuous feature (skew, spread, anything odd), plus a summary
table saved for the report.

In [ ]:
# Histogram grid: one panel per numeric feature.
ncols = 3
nrows = -(-len(numeric) // ncols)  # ceiling division so every feature gets a panel
fig, axes = plt.subplots(nrows, ncols, figsize=(13, nrows * 3))
for ax, col in zip(axes.ravel(), numeric):
    sns.histplot(df[col].dropna(), kde=True, ax=ax)  # kde overlays a smoothed density curve
    ax.set_title(col)
for ax in axes.ravel()[len(numeric):]:  # hide any leftover empty panels
    ax.axis("off")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_hist_numeric.png", dpi=120)

In [ ]:
# Summary statistics table (transposed so each feature is a row).
numeric_summary = df[numeric].describe().T.round(2)
numeric_summary.to_csv(TABLES_DIR / "tbl_eda_numeric_summary.csv")
numeric_summary

## Key features vs readmission

The project plan focuses on `adherence_score`, `bnp`, and `age`. Boxplots split each by readmission
status, and the table reports the group means and their difference. This shows the **direction** of
each association; the formal significance tests are in notebook 03.

In [ ]:
# Required plots: adherence_score and bnp by readmission status (age included as a focus feature).
focus = ["adherence_score", "bnp", "age"]
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, col in zip(axes, focus):
    sns.boxplot(x=df[TARGET], y=df[col], ax=ax)
    ax.set_title(f"{col} by readmission")
    ax.set_xlabel("readmitted_30d")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_focus_by_target.png", dpi=120)

In [ ]:
# Group means by target, with the difference (readmitted minus not). Direction only — not a test.
group_means = df.groupby(TARGET)[focus].mean().T
group_means.columns = ["mean_not_readmitted", "mean_readmitted"]
group_means["difference"] = group_means["mean_readmitted"] - group_means["mean_not_readmitted"]
group_means = group_means.round(3)
group_means.to_csv(TABLES_DIR / "tbl_eda_group_means.csv")
group_means

## Medication indicators vs readmission

For the 0/1 treatment flags, the readmission **rate** within each group is just the mean of the
target. We compare patients on vs off `ace_inhibitor` and `beta_blocker`.

In [ ]:
# Required plot: readmission rate by medication flag (mean of a 0/1 target = the rate).
meds = ["ace_inhibitor", "beta_blocker"]
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, col in zip(axes, meds):
    df.groupby(col)[TARGET].mean().plot.bar(ax=ax, color="indianred")
    ax.set_ylabel("readmission rate")
    ax.set_title(f"readmission rate by {col}")
    ax.set_xticklabels(["off (0)", "on (1)"], rotation=0)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_med_readmit_rate.png", dpi=120)

## Correlation between numeric features

Checks whether any continuous features move together (multicollinearity). If the largest
correlation is tiny, the features are nearly independent — which usually means simple linear
models will do about as well as complex ones, because there is little redundancy or interaction
to exploit.

In [ ]:
corr = df[numeric].corr()  # Pearson correlations among continuous features
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=ax)
ax.set_title("Correlation between numeric features")
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_corr_heatmap.png", dpi=120)

# Largest off-diagonal correlation strength, as a one-number summary.
off_diagonal = corr.where(~np.eye(len(corr), dtype=bool))  # blank out the 1.0 diagonal
print(f"Largest |correlation| between features: {off_diagonal.abs().max().max():.3f}")

## Risk gradients for the strongest numeric signals

We split `adherence_score` and `bnp` into 5 equal-sized bins (`pd.qcut`) and show the readmission
rate in each. A steady climb or fall across bins is a more trustworthy signal than a single odd
spike in one bin.

In [ ]:
# Readmission rate across 5 quantile bins for each feature.
gradient_features = ["adherence_score", "bnp"]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, col in zip(axes, gradient_features):
    bins = pd.qcut(df[col], q=5)                            # 5 bins with roughly equal counts
    rate = df.groupby(bins, observed=True)[TARGET].mean()   # readmission rate per bin
    rate.plot.bar(ax=ax, color="seagreen")
    ax.set_ylabel("readmission rate")
    ax.set_title(f"readmission rate by {col} bin")
    ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(FIGURES_DIR / "fig_risk_gradient.png", dpi=120)

## Confirm the cleaning held

A quick sanity check that we are working on properly cleaned data: `patient_id` is gone and every
numeric value sits inside the plausible clinical ranges defined in notebook 01.
Extreme-but-plausible values are kept on purpose (visible as whiskers and points in the boxplots
above) because they may represent genuine high-risk patients — see notebook 01 for those cleaning
decisions.

In [ ]:
from src.data.clean import RANGES  # the plausible clinical ranges used during cleaning

print("patient_id present?", "patient_id" in df.columns)  # should be False

# For each ranged column, confirm no value falls outside its [low, high] bounds.
for col, (low, high) in RANGES.items():
    if col in df.columns:
        outside = ((df[col] < low) | (df[col] > high)).sum()  # NaNs are ignored by comparisons
        print(f"{col:12s} outside {low}-{high}: {outside}")

## Key takeaways

- **Class balance:** about 41% of patients were readmitted, so the majority-class baseline is
  ~58.9% accuracy. Accuracy alone is a weak metric here — recall and AUROC matter more.
- **Missingness:** concentrated in `creatinine`, `bmi`, and `sodium` (a few percent each). The
  modeling pipeline imputes these after the train/test split.
- **Strongest signals:** higher `bnp` and `age` lean toward readmission; higher `adherence_score`
  leans away from it; being on `ace_inhibitor` or `beta_blocker` is associated with a lower
  readmission rate.
- **Weak, independent features:** pairwise correlations are tiny, so expect modest model
  performance and little advantage for complex models over simple linear ones.
- **Caveat:** these are associations in observational data, not causal effects.